In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# قطع الأسلاك لتقدير قيم التوقع

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*تقدير الاستخدام: 22 ثانية على معالج Heron (ملاحظة: هذا تقدير فحسب. قد يختلف وقت التشغيل الفعلي لديك.)*
## نتائج التعلم
بعد إتمام هذا الدليل التعليمي، يجب أن يفهم المستخدمون:
- كيفية استخدام [`qiskit-addon-cutting`](https://github.com/Qiskit/qiskit-addon-cutting) لتقسيم دائرة كبيرة إلى دوائر فرعية أصغر، مما يقلل من تأثير الضوضاء

## المتطلبات الأساسية
نقترح أن يكون المستخدمون على دراية بالموضوع التالي قبل إتمام هذا الدليل التعليمي:
- استخدام الأولية [Sampler](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)، المستخدمة في هذه العملية

## الخلفية
قطع الدائرة (Circuit-knitting) مصطلح شامل يضم أساليب متعددة لتقسيم الدائرة إلى دوائر فرعية أصغر تتضمن عددًا أقل من البوابات أو الكيوبتات. يمكن تنفيذ كل دائرة فرعية باستقلالية، وتُحصل على النتيجة النهائية عبر معالجة كلاسيكية لاحقة لمخرجات كل دائرة فرعية. يمكن الوصول إلى هذه التقنية من خلال [إضافة Qiskit لقطع الدوائر](https://qiskit.github.io/qiskit-addon-cutting/index.html)؛ راجع [التوثيق](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) إلى جانب [مواد تمهيدية](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html) أخرى للاطلاع على شرح مفصّل للتقنية.

يُركّز هذا الدليل التعليمي على أسلوب يُعرف بـ **قطع الأسلاك**، حيث تُقسَّم الدائرة على طول السلك [\[1\], \[2\]](#references). تجدر الإشارة إلى أن التقسيم سهل في الدوائر الكلاسيكية لأن المخرج عند نقطة التقسيم يمكن تحديده بصورة حتمية إما 0 أو 1. غير أن حالة الكيوبت عند نقطة القطع هي في العموم حالة مختلطة. لذلك، يجب قياس كل دائرة فرعية مرات عدة في أسس مختلفة (عادةً أساس مكتمل طوموغرافيًا كأساس باولي [\[3\], \[4\]](#references))، مع إعداد الحالة المقابلة في حالتها الذاتية. يوضح الشكل أدناه (بإذن من: [\[7\]](#references)) مثالًا على قطع الأسلاك لحالة GHZ ذات أربعة كيوبتات إلى ثلاث دوائر فرعية. هنا، $M_j$ يشير إلى مجموعة من الأسس (عادةً باولي X وY وZ)، و$P_i$ يشير إلى مجموعة من الحالات الذاتية (عادةً $|0\rangle$ و$|1\rangle$ و$|+\rangle$ و$|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

نظرًا لأن كل دائرة فرعية تحتوي على عدد أقل من الكيوبتات والبوابات، فمن المتوقع أن تكون أقل عرضةً للضوضاء. يوضح هذا الدليل التعليمي مثالًا يمكن فيه استخدام هذه الطريقة لتخفيف الضوضاء في النظام بشكل فعال.
## المتطلبات
قبل البدء في هذا الدليل التعليمي، تأكد من تثبيت ما يلي:

- Qiskit SDK الإصدار 2.0 أو أحدث، مع دعم [التصور البياني](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime الإصدار 0.22 أو أحدث ( `pip install qiskit-ibm-runtime` )
- إضافة Qiskit لقطع الدوائر الإصدار 0.10.0 أو أحدث (`pip install qiskit-addon-cutting`)
- Qiskit addon utils الإصدار 0.3 أو أحدث (`pip install qiskit-addon-utils`)
- Qiskit Aer (`pip install qiskit-aer` )
## الإعداد

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## مثال على محاكي صغير النطاق
ينفّذ هذا الدليل التعليمي [نمطًا من أنماط Qiskit](/guides/intro-to-patterns) لمحاكاة دائرة Many-Body Localization (MBL) أحادية البُعد (1D). دائرة MBL هي دائرة فعّالة على مستوى الأجهزة، وهي معلّمة بمعاملين هما $\theta$ و$\vec{\phi}$. عندما يُضبط $\theta$ على $0$ وتُحضَّر الحالة الابتدائية في $|0\rangle$ لجميع الكيوبتات، تكون قيمة التوقع المثالية لـ $\langle Z_i \rangle$ هي $+1$ لكل موقع كيوبت $i$ بصرف النظر عن قيم $\vec{\phi}$. مزيد من التفاصيل حول هذه الدائرة متاح في هذه [المقالة](https://www.nature.com/articles/s41467-025-57623-x).

لاحظ أنه في محاكٍ خالٍ من الضوضاء، ستكون قيمة التوقع المحصّلة مع قطع الدائرة وبدونه متطابقة.
### الخطوة 1: تحويل المدخلات الكلاسيكية إلى مسألة كمومية
#### بناء دائرة MBL أحادية البُعد
نقدّم أولًا دالةً لبناء دائرة MBL أحادية البُعد.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

نحسب متوسط قيمة التوقع $O = \frac{1}{n} \sum_i Z_i$ على جميع الكيوبتات عند $\theta = 0$. بما أن قيمة التوقع المثالية لـ $\langle Z_i \rangle = 1$ $\forall$ $i$، فإن قيمة التوقع المثالية لـ $O$ هي أيضًا $1$. تُختار المعاملات $\phi$ عشوائيًا.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

يجب تعليم الدائرة عبر إدراج **CutWire** في المواضع المطلوبة لتقسيمها. في هذا الدليل التعليمي، نختار تقسيمًا متساويًا. تم تصميم دائرة MBL بحيث يُدرج الإعداد `use_cut=True` في الدالة التعليم بشكل صحيح بعد $\frac{n}{2}$ كيوبت، حيث $n$ هو عدد الكيوبتات في الدائرة الأصلية. كما قمنا بتعيين المعاملات المُولّدة عشوائيًا للدائرة.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif)

### الخطوة 2: تحسين المسألة لتنفيذها على الأجهزة الكمومية
#### قطع الدائرة إلى دوائر فرعية أصغر
الآن نقسّم الدائرة إلى دائرتين فرعيتين أصغر باستخدام [`qiskit-addon-cutting`](https://qiskit.github.io/qiskit-addon-cutting/). يُضيف `qiskit-addon-cutting` بوابة `Move` افتراضية لتقسيم موقع قطع السلك عبر تعديل عدد الكيوبتات بالشكل المناسب. ننشئ الآن الدائرة بهذه البوابة الافتراضية. بما أن هناك قطعة سلك واحدة، سيزداد عدد الكيوبتات المرتبطة بها بمقدار 1.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif)

#### بناء الرصيد وتوسيعه
الرصيد، كما هو محدد سابقًا، سيكون متوسط $Z$ على كل كيوبت. غير أنه بعد إدراج بوابة `Move` الافتراضية، يزداد العدد الفعلي للكيوبتات في الدائرة. يجب توسيع الرصيد وفقًا لذلك لمراعاة هذا التغيير في عدد الكيوبتات. لاحظ أن الرصيد يعمل دائمًا بصورة تافهة (أي $I$) على الكيوبت الإضافي المُضاف لبوابة `Move` الافتراضية.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

غير أن عدد الكيوبتات في الدائرة قد ازداد بعد إدراج عمليات `Move` الافتراضية ذات الكيوبتين عقب القطع. لذا، يجب توسيع الرصيد أيضًا عبر إدراج هويات لمطابقة الدائرة الحالية.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

إليك تصور للدائرتين الفرعيتين:

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif)

يستلزم توسيع الرصيد باستخدام عملية `Move` بنية بيانات `PauliList`. لإعادة بناء قيمة توقع الدائرة الأصلية، نحتاج إلى الرصيد بصيغة `SparsePauliOp`.

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


كما نوقش سابقًا، يجب قياس الدائرة الأعلية عند كل قطعة في أساس باولي، وتُحضَّر الدائرة الأدنى في الحالة الذاتية للأساس. تُنشئ الدالة `generate_cutting_experiments` جميع هذه الدوائر الضرورية والمعاملات المرتبطة بكل دائرة المطلوبة لإعادة البناء. تفاصيل إضافية متاحة في [هذه الورقة البحثية](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

### Step 4: Post-process and return result in desired classical format

Now we retrieve the result of each subexperiment run and reconstruct the expectation value of the uncut circuit:

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

### الخطوة 4: المعالجة اللاحقة وإعادة النتيجة بالصيغة الكلاسيكية المطلوبة
الآن نسترد نتيجة كل تجربة فرعية مُنفَّذة ونُعيد بناء قيمة التوقع للدائرة غير المقطوعة:

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Periodic boundary conditions with circuit cutting](/docs/tutorials/periodic-boundary-conditions-with-circuit-cutting)
- [Circuit cutting for depth reduction](/docs/tutorials/depth-reduction-with-circuit-cutting)
</Admonition>

## References


[1] Peng, T., Harrow, A. W., Ozols, M., & Wu, X. (2020). Simulating large quantum circuits on a small quantum computer. Physical review letters, 125(15), 150504.

[2] Tang, W., Tomesh, T., Suchara, M., Larson, J., & Martonosi, M. (2021, April). Cutqc: using small quantum computers for large quantum circuit evaluations. In Proceedings of the 26th ACM International conference on architectural support for programming languages and operating systems (pp. 473-486).

[3]  Perlin, M. A., Saleem, Z. H., Suchara, M., & Osborn, J. C. (2021). Quantum circuit cutting with maximum-likelihood tomography. npj Quantum Information, 7(1), 64.

[4]  Majumdar, R., & Wood, C. J. (2022). Error mitigated quantum circuit cutting. arXiv preprint arXiv:2211.13431.

[5]  Khare, T., Majumdar, R., Sangle, R., Ray, A., Seshadri, P. V., & Simmhan, Y. (2023). Parallelizing Quantum-Classical Workloads: Profiling the Impact of Splitting Techniques. In 2023 IEEE International Conference on Quantum Computing and Engineering (QCE) (Vol. 1, pp. 990-1000). IEEE.

[6]  Bhoumik, D., Majumdar, R., Saha, A., & Sur-Kolay, S. (2023). Distributed Scheduling of Quantum Circuits with Noise and Time Optimization. arXiv preprint arXiv:2309.06005.

[7]  Majumdar, R. (2024). Efficient Reduction of Resources and Noise in Discrete Quantum Computing Circuits (Doctoral dissertation, Indian Statistical Institute - Kolkata). https://www.proquest.com/openview/b481def90b1cc80e6b58a77c99e8385c/1?pq-origsite=gscholar&cbl=2026366&diss=y